In [ ]:
# Script for styling the Jupyter Notebook using an external CSS file
from IPython.display import HTML

with open("style.css", "r") as f:
    css = f.read()

HTML(f"<style>{css}</style>")

# GraphConv model training

This notebook generates a training report for the GraphConv model, covering three experiments:

1. Hyperparameter optimization — selecting the best-performing model configuration.
2. Sample size optimization — quantifying the minimum number of samples needed to improve accuracy.
3. Full-sample model — training with all available data.

Results directory: `/home/mriveraceron/glv-research/tuning_results/91074c4e25b4`

Data directory: `/home/mriveraceron/glv-research/data_tensors/91074c4e25b4`

# Hyperparameter optimization

In [ ]:
# Display results table
import pandas as pd
from IPython.display import display, HTML

# Experiment directory
dir = '/home/mriveraceron/glv-research/tuning_results/91074c4e25b4'
df = pd.read_csv(f'{dir}/tuning_results.csv')
display(df.sort_values(by="pearson_corr", ascending=False)
          .style.set_table_styles([{'selector': '', 'props': [('margin', '0 auto')]}]))


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
metrics = ['accuracy_idx', 'pearson_corr', 'spearman_corr']
titles  = ['Precisión', 'Correlación de Pearson', 'Correlación de Spearman']
lr_labels = np.sort(df['learning_rate'].unique())[::-1]  # descending
lr_map    = {lra: i for i, lra in enumerate(lr_labels)}
# Legend handles — built once, shared across all columns
combos  = df[['channels', 'layers']].drop_duplicates().reset_index(drop=True)
palette  = plt.cm.tab10.colors
offsets  = np.linspace(-0.2, 0.2, len(combos))
color_map  = {(int(r['channels']), int(r['layers'])): palette[k] for k, (_, r) in enumerate(combos.iterrows())}
offset_map = {(int(r['channels']), int(r['layers'])): offsets[k] for k, (_, r) in enumerate(combos.iterrows())}
handles = [
    mpatches.Patch(color=palette[k], label=f"n={int(r['channels'])} | c={int(r['layers'])}")
    for k, (_, r) in enumerate(combos.iterrows())
]

plt.clf()
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
for ax, metric, title, letter in zip(axes, metrics, titles, ['A', 'B', 'C']):
    for _, row in df.iterrows():
        if pd.isna(row[metric]):
            row[metric] = 0
        key   = (int(row['channels']), int(row['layers']))
        color = color_map[key]
        x     = lr_map[row['learning_rate']] + offset_map[key]
        # Plot line
        ax.plot([x, x], [0, row[metric]], color=color, lw=2, alpha=0.6)
        # Plot circle
        ax.scatter(x, row[metric], color=color, s=100, zorder=5, edgecolors='white', linewidths=0.8)
        # Value number
        ax.text(x, row[metric] + 0.01, f"{row[metric]:.2f}", ha='center', va='bottom', fontsize=7, color=color)
    # Set up x axis
    ax.set_xticks(range(len(lr_labels)))
    ax.set_xticklabels([f"{lra:.0e}" for lra in lr_labels], fontsize=9)
    ax.set_xlabel('Tasa de aprendizaje', fontsize=11, labelpad=10)
    ax.set_ylim(-0.25, df[metric].max() * 1.15)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    # Remove top and right border
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # Add legend at top right
    ax.legend(handles=handles, title='Neuronas | Capas', loc='upper right',
              frameon=True, framealpha=0.4, fontsize=9, title_fontsize=10)
    ax.text(-0.05, 1.05, letter, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='bottom')


plt.suptitle('Desempeño del modelo por tasa de aprendizaje', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])  # leaves 5% headroom at the top for the suptitle
plt.savefig('/home/mriveraceron/glv-research/tuning_results/91074c4e25b4/variants_panel.png')
plt.show()

## Caption
Figure 1. Model performance across hyperparameter configurations (channels, layers, and learning rate). (A) Positive predictive value, (B) Pearson correlation coefficient, and (C) Spearman correlation coefficient. Column boosting method was useed for testing different model settings.

## Interpretation
Based on Figure 1, the optimal hyperparameter configuration corresponds to a learning rate of 1e-3 and 64 channels, with either 5 or 10 layers yielding comparable performance.

The selected model (64 channels, 5 layers, lr = 1e-3) was chosen on the basis of its Spearman 
correlation, where it outperforms all other configurations. Although this model does not achieve the highest precision, halving the number of layers relative to the 10-layer counterpart reduces computational cost without a meaningful loss in predictive performance.



# Sample size optimization

Results directory: `/home/mriveraceron/glv-research/tuning_results/GraphConv_ss`

Data directory: `/home/mriveraceron/glv-research/data_null`

In [ ]:
# Display results table
import pandas as pd
from IPython.display import display, HTML

# Experiment directory
dir = '/home/mriveraceron/glv-research/tuning_results/GraphConv_ss_V2'
df = pd.read_csv(f'{dir}/tuning_results.csv')
display(df.sort_values(by="pearson_corr", ascending=False)
          .style.set_table_styles([{'selector': '', 'props': [('margin', '0 auto')]}]))

In [ ]:
# Elapsed time plot
import matplotlib.pyplot as plt

plt.clf()
plt.close('all')

# --- Global style ---
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.color':       '#e0e0e0',
    'grid.linewidth':   0.8,
    'grid.alpha':       0.7,
})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#fafafa')

# ── Elapsed time plot ──────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#fafafa')

x = df['train_size']
y = df['elapsed(s)']

ax.plot(x, y, color='#2166ac', linewidth=2.2, zorder=3)
ax.fill_between(x, y, alpha=0.08, color='#2166ac')       # subtle area fill
ax.scatter(x, y, color='#2166ac', s=50, zorder=4)        # markers at each point

ax.set_xlabel('Tamaño de muestra para entrenamiento', labelpad=8, fontsize=10)
ax.set_ylabel('Tiempo transcurrido (segundos)',        labelpad=8, fontsize=10)

# ── Metrics plot ───────────────────────────────────────────────
ax = axes[1]
ax.set_facecolor('#fafafa')

x     = df['train_size'].astype(str)
x_pos = range(len(x))

colors          = ['#E69F00', '#56B4E9', '#009E73']
metrics_to_plot = {'ppv_idx':      'VPP',
                   'pearson_corr': 'Corr. Pearson',
                   'spearman_corr':'Corr. Spearman'}

n       = len(metrics_to_plot)
width   = 0.25
offsets = [-width, 0, width]

for (m, lab), col, offset in zip(metrics_to_plot.items(), colors, offsets):
    bars = ax.bar(
        [p + offset for p in x_pos], df[m],
        width=width, label=lab, color=col,
        alpha=0.88, edgecolor='white', linewidth=0.6,
        zorder=3
    )
    ax.bar_label(bars, fmt='%.2f', fontsize=7.5, padding=3,rotation=90, color='#333333')

ax.set_xticks(x_pos)
ax.set_xticklabels(x, rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Tamaño de muestra para entrenamiento', labelpad=8, fontsize=10)
ax.set_ylabel('Valor',                                labelpad=8, fontsize=10)
ax.set_ylim(0, ax.get_ylim()[1] * 1.18)               # headroom for bar labels
ax.legend(
    title='Rendimiento del modelo',
    loc='upper right', frameon=True,
    framealpha=0.5, edgecolor='#cccccc',
    fontsize=9, title_fontsize=10
)

# ── Title & layout ─────────────────────────────────────────────
for ax, letter in zip(axes, ['A', 'B']):
    ax.text(-0.05, 1.05, letter, transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom', ha='right')


plt.suptitle('Desempeño de la mejor variante de GraphConv por tamaño de muestra',
             fontsize=14, fontweight='bold', color='#222222', y=1.01)
plt.tight_layout()
save_path = '/home/mriveraceron/glv-research/tuning_results/GraphConv_ss_V2/panel_img.png'
print(f'Image saved at: {save_path}')
plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor="white")
plt.show()

## Caption
Figure 2. Model performance across training sample sizes. (A) Elapsed training time as a function of training size. (B) Predictive performance metrics (VPP, Pearson and Spearman correlations) across training sizes. Random networks were employed for training.

## Interpretation
As shown in Figure 2, elapsed training time increases linearly with sample size. Predictive performance improves with larger training sets but eventually plateaus, suggesting that beyond a certain threshold, additional samples yield diminishing returns in model accuracy.
